# Prescribe a current profile

Alternatively to fixing the $\iota(s)$ profile, one can also fix the total toroidal current $I_{\text{tor}}(s)$ when solving for an equilibrium. This notebook showcases how to do so in GVEC.

We start with the necessary imports and set the number of threads to use:

In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np

# OMP number of threads for gvec run needs to be before import of gvec
os.environ["OMP_NUM_THREADS"] = "2"
# needs `pip install` of gvec in virtual environment, and to be run in that environment!!!
import gvec

In this tutorial, we load the parameters from the `.toml` file written in the [elliptic stellarator tutorial](./020_stellarator).

In [ ]:
params = gvec.util.read_parameters("ellipstell_parameters.toml")

## New parameters 

Running GVEC with a prescribed $I_{\text{tor}}$ slightly changes the underlying algorithm. However, the basic syntax in terms of the parameters is almost the same as in the fixed $\iota$ case.

The basic steps we need are:
- prescribe $I_{\text{tor}(s)}$ profile via `I_tor`, with the same syntax to the one used for `iota` and `pres` profiles.
- set the new parameter `picard_current`
- *optionally* provide an initial guess for $\iota$

For our stellarator example we choose to set `I_tor` to zero and utilize the automatic setup for `picard_current` by setting `picard_current="auto"`. 

Note that prescribing a toroidal current profile will *always* treat `iota` as an initial guess!



In [ ]:
# toroidal current profile (=0)
params["I_tor"] = {
    "type": "polynomial",
    "coefs": [0.0],
}
# initial guess for iota (=0)
params["iota"] = {"type": "polynomial", "coefs": [0.0]}

params["picard_current"] = "auto"
params["minimize_tol"] = 1.0e-5
params["totalIter"] = 100000

---

## Running GVEC with prescribed toroidal current

Next we, run GVEC and visualize again the minimization diagnostics:

In [ ]:
runpath = "run_current_profile"

run = gvec.run(params, runpath=runpath, keep_intermediates="stages")

fig = run.plot_diagnostics_minimization()

### Understanding the diagnostics

As we can see the screen output as well as the visualization has undergone some changes in comparison to the fixed $\iota$ run. We mentioned above that the underlying algorithm is slightly different. So, what happened here?
- GVEC optimizes for $I_{\text{tor}}$ by adapting $\iota$ using Picard iterations
- $\iota$ can be split into two major contributions:
    - one that solely originates from the geometry: $\iota_0$
    - one that also includes contributions from $I_{\text{tor}}$: $\iota_{\text{curr}}$
    - $\iota_T = \iota_{\text{curr}} + \iota_0$
- GVEC always runs with fixed $\iota$ for several iterations before updating $\iota$ again due to the $\iota_{\text{curr}}$ contribution
- changing the geometry will lead to a discrepancy between $\iota_T$ and the currently specified $\iota$:
    - we track the discrepancy via $\Delta\iota_{\text{rms}}$
- after a $\iota$ update GVEC restarts from its previous state with the new profile
- `picard_current` is used to control the Picard iterations / strategy:
    - the `"auto"` modus automatically designs `stages` depending on the chosen `minimize_tol`. 
    - the `"auto"` modus is the recommended setting

For more theory behind this approach and more advanced control options see the [stages section in the GVEC user guide](https://gvec.readthedocs.io/latest/user/stages.html).

---

## Visualize the result

Since we optimize for $I_{\text{tor}}$, small deviations from the prescribed profile are possible. Therefore, let us visualize some of the quantities of interest here: 
- the toroidal current contribution to the rotational transform $\iota_{\text{curr}}$ / `iota_curr`
- the final $\iota$ profile / `iota`
- the poloidal current `I_pol`
- the toroidal current `I_tor` calculated using `iota` and the geometry

For more visualization later on we also calculate `p`, `mod_B`, `X1` and `X2`.

In [ ]:
varlist = ["iota", "iota_curr", "I_tor", "I_pol", "p", "mod_B", "X1", "X2"]
zeta = np.linspace(0, 2 * np.pi / run.state.nfp, 9)
ev = run.state.evaluate(*varlist, rho=101, theta=np.linspace(0, 2 * np.pi, 40), zeta=zeta)

First, we have a look at the rotational transform and current profiles.

In [ ]:
# === Profiles === #
fig, axs = plt.subplots(2, 2, figsize=(10, 6), tight_layout=True, sharex=True)

for ax, var in zip(axs.flatten(), ["iota_curr", "iota", "I_tor", "I_pol"]):
    ax.plot(ev.rho, ev[var], label=f"${ev[var].attrs['symbol']}$")
    ax.set(
        title=f"{ev[var].attrs['long_name']}",
        xlabel="$\\rho\\sim$ sqrt(tor. flux)",
        ylabel=f"${ev[var].attrs['symbol']}$",
    )

for ax in axs.flat:
    ax.legend()

As we can see the toroidal current calculated from the equilibrium solution as well as $\iota_{\text{curr}}$ are almost zero as they should be.

Finally, we can also visualize some of the poloidal cross-sections as before.

In [ ]:
# === Cross-sections === #
fig, axs = plt.subplots(3, 3, figsize=(12, 10), sharex=True, sharey=True)

vmin = np.amin(ev.mod_B)
vmax = np.amax(ev.mod_B)
r_levels = np.linspace(0, 1 - 1e-10, 5)
t_levels = np.linspace(0, 2 * np.pi, 9)
for ax, zeta_pos in zip(axs.flat, ev.zeta[:]):
    # select zeta-plane from dataset
    ev_z = ev.sel(zeta=zeta_pos, method="nearest")

    c = ax.contourf(ev_z.X1, ev_z.X2, ev_z.mod_B, vmin=vmin, vmax=vmax)

    ax.contour(ev_z.X1, ev_z.X2, 0 * ev_z.X1 + ev_z.rho, r_levels, colors="white", alpha=0.7)
    ax.contour(ev_z.X1, ev_z.X2, 0 * ev_z.X1 + ev_z.theta, t_levels, colors="white", alpha=0.7)

    ax.set(
        aspect="equal",
        title=f"$\\zeta/2\\pi = {zeta_pos / (2 * np.pi):.3f}$",
    )


for ax in axs[-1, :]:
    ax.set_xlabel(f"${ev.X1.attrs['symbol']}$")
for ax in axs[:, 0]:
    ax.set_ylabel(f"${ev.X2.attrs['symbol']}$")

fig.colorbar(c, ax=axs, shrink=0.5, label="|B|");

## Summary
To summarize, for running GVEC with a prescribed current profile one has to set the parameters `I_tor` and `picard_current`. Since in this case GVEC updates the rotational transform profile using Picard iterations, small deviations from the prescribed current profile can occur. The parameter `picard_current` is utilized to control these Picard iterations. Using `picard_current="auto"` is the recommended approach. Detailed control options can be found in the [stages section in the GVEC user guide](https://gvec.readthedocs.io/latest/user/stages.html).